Run this notebook from its lesson folder. Select this lesson's Python kernel. Copy `.env.example` to `.env` or enter keys at the hidden prompts. Read `README.md` first.


# LiteLLM 101

One small refund assistant, a primary model, and a backup. Call models through a common interface, test fallbacks, and verify which deployment answered.


## Setup

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env", override=False)


### Key, models, and one question

Both models use the same provider to keep setup small. This demonstrates model failover; surviving a provider-wide outage requires a suitable backup on an independent provider.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = (
    os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")
).strip()
if not os.environ["OPENAI_API_KEY"]:
    raise ValueError("An OpenAI API key is required.")

# Use the bundled model metadata; this lesson does not need a live price-map download.
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

import litellm
from copy import deepcopy
from IPython.display import Markdown, display

PRIMARY_MODEL = "openai/gpt-4o-mini-2024-07-18"
BACKUP_MODEL = "openai/gpt-4.1-nano-2025-04-14"
MESSAGES = [
    {"role": "system", "content": "Our fictional policy allows refunds within 30 days of purchase. Answer using only this policy, in one short sentence."},
    {"role": "user", "content": "What is the refund window?"},
]

## 1. One interface for model calls

`completion()` accepts a provider/model name and messages. The response follows a common format. Changing the model does not require a new calling pattern, although credentials and supported features can differ.

In [ ]:
response = litellm.completion(
    model=PRIMARY_MODEL, messages=MESSAGES,
    temperature=0, max_tokens=80, timeout=45, num_retries=0,
)
print(response.choices[0].message.content)
print("Provider model:", response.model)

## 2. A primary model and a backup
### Routing rules

`primary` and `backup` are names our app uses. Each points to a provider model. A **retry** makes another attempt within the requested model group. A **fallback** moves to a configured backup group.

The rule below allows one retry, then a fallback from `primary` to `backup`. Each group has one deployment in this example.

In [ ]:
MODEL_LIST = [
    {
        "model_name": "primary",
        "litellm_params": {"model": PRIMARY_MODEL, "api_key": "os.environ/OPENAI_API_KEY"},
        "model_info": {"id": "primary-deployment"},
    },
    {
        "model_name": "backup",
        "litellm_params": {"model": BACKUP_MODEL, "api_key": "os.environ/OPENAI_API_KEY"},
        "model_info": {"id": "backup-deployment"},
    },
]
ROUTER_SETTINGS = {
    "fallbacks": [{"primary": ["backup"]}],
    "num_retries": 1,
    "retry_after": 0,
    "disable_cooldowns": True,  # Keep independent teaching tests repeatable.
}

router = litellm.Router(model_list=deepcopy(MODEL_LIST), **ROUTER_SETTINGS)

### A healthy request

The app requests `primary`. The deployment ID tells us which route actually served the response. It is stronger evidence than asking the model to identify itself.

In [ ]:
response = router.completion(
    model="primary", messages=MESSAGES, temperature=0, max_tokens=80, timeout=45,
)
served_by = response._hidden_params["model_id"]
print(response.choices[0].message.content)
print("Requested: primary | Served by:", served_by)
assert served_by == "primary-deployment", "Primary did not serve this request; check model access."

## 3. Fallback scenarios
### Controlled errors, real backup answers

The next cell injects an error into **only the primary deployment** of a fresh test Router. The backup still makes a real model call. The app keeps requesting `primary`.

| Scenario | What we simulate |
|---|---|
| Rate limit | The primary cannot accept another request. |
| Timeout | The primary does not respond within the allowed time. |
| Provider unavailable | The primary service cannot handle the request. |

`mock_response` is a test control. The timeout error is injected immediately, so we do not wait for an actual timeout. These tests exercise the Router's failure handling, not provider network behavior.

In [ ]:
SCENARIOS = [
    ("Rate limit", litellm.RateLimitError),
    ("Timeout", litellm.Timeout),
    ("Provider unavailable", litellm.ServiceUnavailableError),
]
results = []

for scenario, error_type in SCENARIOS:
    test_router = litellm.Router(model_list=deepcopy(MODEL_LIST), **ROUTER_SETTINGS)
    # Add the test exception after initialization; exception objects cannot be deep-copied.
    test_router.model_list[0]["litellm_params"]["mock_response"] = error_type(
        message=f"Simulated: {scenario}", model=PRIMARY_MODEL, llm_provider="openai",
    )
    response = test_router.completion(
        model="primary", messages=MESSAGES, temperature=0, max_tokens=80, timeout=45,
    )
    served_by = response._hidden_params["model_id"]
    assert served_by == "backup-deployment", f"Fallback was not verified for {scenario}."
    results.append((scenario, served_by))
    print(f"{scenario}: primary failed; served by {served_by}")
    print(response.choices[0].message.content, "\n")

display(Markdown(
    "| Scenario | Requested | Served by |\n|---|---|---|\n" +
    "\n".join(f"| {scenario} | primary | {deployment} |" for scenario, deployment in results)
))

**Insight:** our application requested the same name every time; the Router followed the backup rule. The deployment IDs verify the change. A fallback improves availability, but the backup can also fail or produce a different answer. It does not fix a quality problem in an otherwise successful response.

## 4. What to take into your project
Use `completion()` for a common model interface and `Router` when you need a primary and backup. Verify the served deployment rather than asking the model to identify itself. These controlled failures test routing behavior; evaluate answer quality separately.
